# 02-yelp-gpu-training — Phase 3–5: Transformer Training trên Kaggle GPU

**Quy trình:** Đồng bộ dữ liệu → Tạo configs → Smoke Test & Short Run → Huấn luyện BERT (Huber vs MSE) & Chốt Loss → Huấn luyện DistilBERT / RoBERTa → Dự đoán Test 1 lần → Xuất `model_summary.csv` & Kiểm thử tích hợp (`validate_outputs.py`).

**Trước khi chạy:**
- Bật **Accelerator → GPU T4 x2** (hoặc **P100**).
- Bật **Internet** (clone GitHub và tải pretrained weights từ Hugging Face).
- **Add Input → Notebooks → Your Work**: gắn **`01-yelp-data-eda-baseline` (Version 6)** — Kaggle sẽ tự động mount các file `.parquet`, `split_stats.json`, `baseline_val.pkl` và `baseline_test.csv` vào `/kaggle/input`.


In [1]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch

def run_cmd(cmd: str, check: bool = True) -> int:
    """Chạy lệnh shell và in log theo thời gian thực ra output cell."""
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if check and ret != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {ret}): {cmd}")
    return ret

print("=" * 60)
print("CELL 1 — Kiểm tra GPU, Clone Repo & Đồng bộ dữ liệu")
print("=" * 60)

# 1. Kiểm tra GPU
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Chưa bật GPU! Vào Session options -> Accelerator -> chọn GPU T4 x2 hoặc P100."
    )
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM tổng: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# 2. Clone / Pull repo
REPO_DIR = "/kaggle/working/topic"
if not os.path.exists(REPO_DIR):
    print("\n📥 Đang clone repo...")
    run_cmd(f"git clone https://github.com/toan58936/topic.git {REPO_DIR}")
else:
    print("\n🔄 Đang pull repo...")
    run_cmd(f"git -C {REPO_DIR} pull")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# 3. Cài dependencies
print("\n📦 Đang cài requirements.txt...")
run_cmd("pip install -q -r requirements.txt")

# 4. Đồng bộ dữ liệu từ Output Notebook 1 (tránh SameFileError)
DATA_DIR = Path("/kaggle/working/data")
PREDS_DIR = Path("/kaggle/working/results/preds")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

required_data = ["train.parquet", "val.parquet", "test.parquet", "split_stats.json"]
has_data = all((DATA_DIR / f).exists() for f in required_data)

if has_data:
    print("📦 Dữ liệu đã có sẵn tại /kaggle/working/data")
else:
    for root, _, _ in os.walk("/kaggle/input"):
        root_p = Path(root)
        if all((root_p / f).exists() for f in required_data):
            for f in required_data + ["split_id.json"]:
                src_f = root_p / f
                if src_f.exists():
                    shutil.copy2(src_f, DATA_DIR / f)
            has_data = True
            print("📦 Đã copy data từ:", root_p)
            break

if not has_data:
    raise FileNotFoundError(
        "Không tìm thấy bộ file .parquet trong /kaggle/input hoặc /kaggle/working/data. "
        "Hãy Add Input notebook 01-yelp-data-eda-baseline (Version 6)."
    )

has_base = (PREDS_DIR / "baseline_val.pkl").exists() and (PREDS_DIR / "baseline_test.csv").exists()
if has_base:
    print("📦 Baseline artifacts đã có sẵn tại /kaggle/working/results/preds")
else:
    for root, _, _ in os.walk("/kaggle/input"):
        root_p = Path(root)
        if (root_p / "baseline_val.pkl").exists() and (root_p / "baseline_test.csv").exists():
            shutil.copy2(root_p / "baseline_val.pkl", PREDS_DIR / "baseline_val.pkl")
            shutil.copy2(root_p / "baseline_test.csv", PREDS_DIR / "baseline_test.csv")
            has_base = True
            print("📦 Đã copy baseline từ:", root_p)
            break

if not has_base:
    raise FileNotFoundError(
        "Không tìm thấy baseline_val.pkl và baseline_test.csv. "
        "Hãy kiểm tra Output của Notebook 1 đã được gắn vào Input chưa."
    )

# 5. Đọc split_stats
split_stats = json.loads((DATA_DIR / "split_stats.json").read_text(encoding="utf-8"))
SPLIT_ID = split_stats["split_id"]
print("\n✅ Sẵn sàng!")
print("   Split ID:", SPLIT_ID)
print("   Train/Val/Test:", split_stats["train_size"], split_stats["val_size"], split_stats["test_size"])
print("   Thư mục làm việc:", os.getcwd())


CELL 1 — Kiểm tra GPU, Clone Repo & Đồng bộ dữ liệu
CUDA available: True
GPU: Tesla T4
VRAM tổng: 14.6 GB

🔄 Đang pull repo...
Already up to date.

📦 Đang cài requirements.txt...
📦 Dữ liệu đã có sẵn tại /kaggle/working/data
📦 Baseline artifacts đã có sẵn tại /kaggle/working/results/preds

✅ Sẵn sàng!
   Split ID: f0e6776f8a1a
   Train/Val/Test: 42000 9001 8999
   Thư mục làm việc: /kaggle/working/topic


## 2. Tạo thư mục `configs/` và 6 file YAML chuẩn schema `src/train.py`

Mỗi file YAML chỉ dùng đúng các khóa được khai báo trong `DEFAULT_CONFIG` của `src/train.py` (`load_config()` sẽ từ chối khóa lạ bằng `ValueError`).

Tạo 6 file:
- `smoke.yaml` — Smoke Test (16 mẫu, 1 epoch)
- `short_bert.yaml` — Short GPU Run (1 epoch BERT thật, đo tốc độ & VRAM)
- `train_bert_huber.yaml` — BERT Huber (3 epochs)
- `train_bert_mse.yaml` — BERT MSE (3 epochs, đối chứng)
- `train_distilbert.yaml` — DistilBERT (3 epochs, sẽ cập nhật loss sau Phase 4)
- `train_roberta.yaml` — RoBERTa (3 epochs, có điều kiện quota GPU, sẽ cập nhật loss sau Phase 4)


In [2]:
from pathlib import Path
import pandas as pd
import yaml

print("=" * 60)
print("CELL 2 — Tạo configs/ và data_smoke/")
print("=" * 60)

CONFIG_DIR = Path("/kaggle/working/topic/configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = "/kaggle/working/data"
SMOKE_DIR = "/kaggle/working/data_smoke"
CKPT_DIR = "/kaggle/working/checkpoints"
RES_DIR = "/kaggle/working/results"

# --- Tạo bộ dữ liệu nhỏ cho Smoke Test (16 train / 8 val / 8 test) ---
smoke_path = Path(SMOKE_DIR)
smoke_path.mkdir(parents=True, exist_ok=True)
for split, n in [("train", 16), ("val", 8), ("test", 8)]:
    src = Path(f"{DATA_DIR}/{split}.parquet")
    df = pd.read_parquet(src).head(n)
    df.to_parquet(smoke_path / f"{split}.parquet", index=False)
print("📊 Đã tạo data_smoke/: 16 train / 8 val / 8 test")

# --- Base config (khớp DEFAULT_CONFIG của src/train.py) ---
base_cfg = {
    "run_name": "bert",
    "backbone": "bert",
    "pooling": "mean",
    "dropout": 0.1,
    "output_bias_init": "train_mean",
    "loss": "huber",
    "huber_delta": 1.0,
    "seed": 42,
    "max_length": 256,
    "batch_size": 16,
    "epochs": 3,
    "lr": 2.0e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "grad_clip": 1.0,
    "fp16": True,
    "num_workers": 0,
    "device": "auto",
    "log_every": 250,
    "data_dir": DATA_DIR,
    "checkpoint_dir": CKPT_DIR,
    "results_dir": RES_DIR,
}

def write_cfg(filename: str, overrides: dict) -> None:
    cfg = dict(base_cfg)
    cfg.update(overrides)
    with open(CONFIG_DIR / filename, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    print(f"   ✅ Ghi {CONFIG_DIR / filename}")

write_cfg("smoke.yaml", {
    "run_name": "smoke_bert",
    "backbone": "bert",
    "epochs": 1,
    "batch_size": 8,
    "log_every": 1,
    "data_dir": SMOKE_DIR,
})

write_cfg("short_bert.yaml", {
    "run_name": "short_bert",
    "backbone": "bert",
    "epochs": 1,
})

write_cfg("train_bert_huber.yaml", {
    "run_name": "bert",
    "backbone": "bert",
    "loss": "huber",
    "epochs": 3,
})

write_cfg("train_bert_mse.yaml", {
    "run_name": "bert_mse",
    "backbone": "bert",
    "loss": "mse",
    "epochs": 3,
})

write_cfg("train_distilbert.yaml", {
    "run_name": "distilbert",
    "backbone": "distilbert",
    "loss": "huber",
    "epochs": 3,
})

write_cfg("train_roberta.yaml", {
    "run_name": "roberta",
    "backbone": "roberta",
    "loss": "huber",
    "epochs": 3,
})

print("\n✅ Đã tạo đủ 6 file cấu hình trong configs/")


CELL 2 — Tạo configs/ và data_smoke/
📊 Đã tạo data_smoke/: 16 train / 8 val / 8 test
   ✅ Ghi /kaggle/working/topic/configs/smoke.yaml
   ✅ Ghi /kaggle/working/topic/configs/short_bert.yaml
   ✅ Ghi /kaggle/working/topic/configs/train_bert_huber.yaml
   ✅ Ghi /kaggle/working/topic/configs/train_bert_mse.yaml
   ✅ Ghi /kaggle/working/topic/configs/train_distilbert.yaml
   ✅ Ghi /kaggle/working/topic/configs/train_roberta.yaml

✅ Đã tạo đủ 6 file cấu hình trong configs/


## 3. Phase 3 — Smoke Test & Short GPU Run

- **Smoke Test**: 16 mẫu train, 8 val, 8 test, 1 epoch — kiểm tra CUDA, `fp16`, không lỗi `NaN`.
- **Short GPU Run**: 1 epoch BERT thật — đo `seconds_per_epoch` để ước tính ngân sách thời gian cho 12 epochs ở Phase 4–5.


In [3]:
import json
import os

print("=" * 60)
print("CELL 3 — Phase 3: Smoke Test & Short GPU Run")
print("=" * 60)

# 3.1 Smoke Test (tự bỏ qua nếu đã completed)
print("\n--- 3.1 Smoke Test (16 mẫu, 1 epoch) ---")
smoke_meta = "/kaggle/working/results/runs/smoke_bert/run_meta.json"
if os.path.exists(smoke_meta) and json.loads(open(smoke_meta, encoding="utf-8").read()).get("status") == "completed":
    print("⏭️  Smoke Test đã completed trước đó, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/smoke.yaml")
print("✅ Smoke Test PASS")

# 3.2 Short GPU Run (1 epoch BERT thật)
print("\n--- 3.2 Short GPU Run (1 epoch BERT thật) ---")
meta_path = "/kaggle/working/results/runs/short_bert/run_meta.json"
if os.path.exists(meta_path) and json.loads(open(meta_path, encoding="utf-8").read()).get("status") == "completed":
    print("⏭️  Short GPU Run đã completed trước đó, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/short_bert.yaml")

# Đọc run_meta.json để lấy seconds_per_epoch
if os.path.exists(meta_path):
    meta_short = json.loads(open(meta_path, "r", encoding="utf-8").read())
    sec_per_ep = meta_short.get("epoch_seconds_mean")
    n_params = meta_short.get("n_params")
    print(f"\n📊 Short Run BERT -> epoch_seconds_mean: {sec_per_ep:.1f}s | n_params: {n_params:,}")
    if sec_per_ep:
        est_12ep_min = sec_per_ep * 12 / 60
        print(f"⏱️  Ước tính 12 epochs (4 runs × 3 epochs): ~{est_12ep_min:.1f} phút")
else:
    print("⚠️  Không tìm thấy run_meta.json của short_bert")

print("\n📌 Trạng thái GPU hiện tại:")
run_cmd("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader", check=False)
print("\n✅ Phase 3 hoàn tất")


CELL 3 — Phase 3: Smoke Test & Short GPU Run

--- 3.1 Smoke Test (16 mẫu, 1 epoch) ---
⏭️  Smoke Test đã completed trước đó, bỏ qua.
✅ Smoke Test PASS

--- 3.2 Short GPU Run (1 epoch BERT thật) ---
⏭️  Short GPU Run đã completed trước đó, bỏ qua.

📊 Short Run BERT -> epoch_seconds_mean: 629.6s | n_params: 108,892,417
⏱️  Ước tính 12 epochs (4 runs × 3 epochs): ~125.9 phút

📌 Trạng thái GPU hiện tại:
Tesla T4, 3 MiB, 15360 MiB
Tesla T4, 3 MiB, 15360 MiB

✅ Phase 3 hoàn tất


## 4. Phase 4 — Huấn luyện BERT Huber vs MSE & Áp dụng Loss Rule (Mốc 1)

Chạy 2 lần huấn luyện đầy đủ 3 epochs trên `bert-base-uncased` cùng seed và siêu tham số:
- Run 1 (`bert`): `loss = huber`, `delta = 1.0`
- Run 2 (`bert_mse`): `loss = mse`

**Loss Rule** (Define Mục 9): Chuyển sang MSE cho toàn bộ giai đoạn sau **nếu và chỉ nếu**:
- `RMSE_MSE <= 0.99 × RMSE_Huber` **và**
- `MAE_MSE <= 1.01 × MAE_Huber`

**Mốc 1:** Val RMSE của BERT thắng `<= 0.85 × RMSE_baseline_val` và Val Macro-MAE `<= Macro_MAE_baseline_val` (tính trực tiếp từ `baseline_val.pkl` trên `val.parquet`).


In [4]:
import json
from pathlib import Path

def _is_completed(run_name: str) -> bool:
    p = Path(f"/kaggle/working/results/runs/{run_name}/run_meta.json")
    if not p.exists():
        return False
    return json.loads(p.read_text(encoding="utf-8")).get("status") == "completed"

print("=" * 60)
print("CELL 4A — Phase 4.1: Train BERT + Huber (3 epochs)")
print("=" * 60)

if _is_completed("bert"):
    print("⏭️  Run 'bert' đã completed, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/train_bert_huber.yaml")
print("✅ Xong BERT + Huber")

CELL 4A — Phase 4.1: Train BERT + Huber (3 epochs)
⏭️  Run 'bert' đã completed, bỏ qua.
✅ Xong BERT + Huber


In [5]:
import json
from pathlib import Path
import pandas as pd
import yaml

from src.baseline import load_baseline, predict_baseline
from src.data_setup import create_dataloaders
from src.engine import compare_runs, evaluate
from src.metrics import compute_metrics, per_star_metrics, threshold_moc1
from src.model_builder import build_tokenizer
from src.predict import load_trained_model

def _is_completed(run_name: str) -> bool:
    p = Path(f"/kaggle/working/results/runs/{run_name}/run_meta.json")
    if not p.exists():
        return False
    return json.loads(p.read_text(encoding="utf-8")).get("status") == "completed"

print("=" * 60)
print("CELL 4B — Phase 4.2: Train BERT + MSE & Loss Rule (Mốc 1)")
print("=" * 60)

if _is_completed("bert_mse"):
    print("⏭️  Run 'bert_mse' đã completed, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/train_bert_mse.yaml")

cmp_df = compare_runs(["bert", "bert_mse"], "/kaggle/working/results")
print("\n--- So sánh Huber vs MSE trên tập Val ---")
print(cmp_df.to_string(index=False))

huber_row = cmp_df.loc[cmp_df["run_name"] == "bert"].iloc[0]
mse_row = cmp_df.loc[cmp_df["run_name"] == "bert_mse"].iloc[0]

rmse_huber, mae_huber = float(huber_row["best_val_rmse"]), float(huber_row["best_val_mae"])
rmse_mse, mae_mse = float(mse_row["best_val_rmse"]), float(mse_row["best_val_mae"])

switch_to_mse = (rmse_mse <= 0.99 * rmse_huber) and (mae_mse <= 1.01 * mae_huber)
CHOSEN_LOSS = "mse" if switch_to_mse else "huber"
BERT_WINNER_RUN = "bert_mse" if switch_to_mse else "bert"
BEST_BERT_VAL_RMSE = rmse_mse if switch_to_mse else rmse_huber

val_df = pd.read_parquet("/kaggle/working/data/val.parquet")
base_art = load_baseline("/kaggle/working/results/preds/baseline_val.pkl")
base_val_pred = predict_baseline(base_art["pipeline"], val_df["text"].astype(str).tolist())
base_val_m = compute_metrics(val_df["stars"], base_val_pred)
base_val_ps = per_star_metrics(val_df["stars"], base_val_pred)

MOC1_RMSE_THRESHOLD = float(threshold_moc1(base_val_m["rmse"]))
MOC1_MACRO_MAE_THRESHOLD = float(base_val_ps["macro_mae"])

tok_bert = build_tokenizer("bert")
_, val_loader_bert, _ = create_dataloaders("/kaggle/working/data", tok_bert, batch_size=32, num_workers=0)
bert_model, _ = load_trained_model(f"/kaggle/working/checkpoints/{BERT_WINNER_RUN}/best_model.pt", device="cuda")
val_eval = evaluate(bert_model, val_loader_bert, "cuda")
bert_val_macro_mae = float(per_star_metrics(val_eval["y_true"], val_eval["y_pred_raw"])["macro_mae"])
del bert_model

print(f"\n⚖️  CHOSEN_LOSS = '{CHOSEN_LOSS}' | BERT_WINNER_RUN = '{BERT_WINNER_RUN}'")
print(f"🎯 Mốc 1 Val RMSE:      {BEST_BERT_VAL_RMSE:.4f} (ngưỡng <= {MOC1_RMSE_THRESHOLD:.4f})")
print(f"🎯 Mốc 1 Val Macro-MAE: {bert_val_macro_mae:.4f} (ngưỡng <= {MOC1_MACRO_MAE_THRESHOLD:.4f})")

decision_payload = {
    "chosen_loss": CHOSEN_LOSS,
    "bert_winner_run": BERT_WINNER_RUN,
    "rmse_huber": rmse_huber,
    "mae_huber": mae_huber,
    "rmse_mse": rmse_mse,
    "mae_mse": mae_mse,
    "best_bert_val_rmse": BEST_BERT_VAL_RMSE,
    "bert_val_macro_mae": bert_val_macro_mae,
    "moc1_rmse_threshold": MOC1_RMSE_THRESHOLD,
    "moc1_macro_mae_threshold": MOC1_MACRO_MAE_THRESHOLD,
}
Path("/kaggle/working/results/phase4_decision.json").write_text(
    json.dumps(decision_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)

for bb in ["distilbert", "roberta"]:
    cfg_path = Path(f"/kaggle/working/topic/configs/train_{bb}.yaml")
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    cfg["loss"] = CHOSEN_LOSS
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("✅ Đã chốt Phase 4 & cập nhật configs/train_distilbert.yaml, train_roberta.yaml")

CELL 4B — Phase 4.2: Train BERT + MSE & Loss Rule (Mốc 1)
⏭️  Run 'bert_mse' đã completed, bỏ qua.

--- So sánh Huber vs MSE trên tập Val ---
run_name  best_epoch  best_val_rmse  best_val_mae  train_seconds  epoch_seconds_mean  rel_diff_pct_vs_first
    bert           3       0.567337      0.343471    1787.610766          595.870255               0.000000
bert_mse           3       0.562778      0.345867    1797.072555          599.024185               0.803549


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


⚖️  CHOSEN_LOSS = 'huber' | BERT_WINNER_RUN = 'bert'
🎯 Mốc 1 Val RMSE:      0.5673 (ngưỡng <= 0.6623)
🎯 Mốc 1 Val Macro-MAE: 0.4452 (ngưỡng <= 0.6329)
✅ Đã chốt Phase 4 & cập nhật configs/train_distilbert.yaml, train_roberta.yaml


## 5. Phase 5a — Huấn luyện DistilBERT → RoBERTa

Sử dụng đúng `CHOSEN_LOSS` đã chốt ở Bước 4, giữ nguyên toàn bộ thiết lập còn lại.

- `distilbert` (`distilbert-base-uncased`, 3 epochs): Chạy trước vì nhẹ (~66.4M tham số).
- `roberta` (`roberta-base`, 3 epochs): Có điều kiện quota GPU. Nếu không đủ, đổi `RUN_ROBERTA = False` để bỏ hẳn (không giảm epoch/batch size).


In [6]:
import json
from pathlib import Path

def _is_completed(run_name: str) -> bool:
    p = Path(f"/kaggle/working/results/runs/{run_name}/run_meta.json")
    if not p.exists():
        return False
    return json.loads(p.read_text(encoding="utf-8")).get("status") == "completed"

print("=" * 60)
print("CELL 5A — Phase 5.1: Train DistilBERT (3 epochs)")
print("=" * 60)

if _is_completed("distilbert"):
    print("⏭️  Run 'distilbert' đã completed, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/train_distilbert.yaml")
print("✅ Xong DistilBERT")

CELL 5A — Phase 5.1: Train DistilBERT (3 epochs)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2904.90it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  epoch 1 step 250/250+ loss=0.2497 (8.3 it/s)
  epoch 1 step 500/500+ loss=0.1900 (8.3 it/s)
  epoch 1 step 750/750+ loss=0.1533 (8.1 it/s)
  epoch 1 step 1000/1000+ loss=0.1020 (8.1 it/s)
  epoch 1 step 1250/1250+ loss=0.1135 (8.1 it/s)
  epoch 1 step 1500/1500+ loss=0.3026 (8.0 it/s)
  epoch 1 step 1750/1750+ loss=0.2142 (8.0 it/s)
  epoch 1 step 2000/2000+ loss=0.1961 (8.0 it/s)


In [7]:
import json
from pathlib import Path

def _is_completed(run_name: str) -> bool:
    p = Path(f"/kaggle/working/results/runs/{run_name}/run_meta.json")
    if not p.exists():
        return False
    return json.loads(p.read_text(encoding="utf-8")).get("status") == "completed"

RUN_ROBERTA = True  # Đổi thành False nếu muốn bỏ qua RoBERTa

print("=" * 60)
print("CELL 5B — Phase 5.2: Train RoBERTa (3 epochs)")
print("=" * 60)

if not RUN_ROBERTA:
    print("⏭️  Bỏ qua RoBERTa theo thiết lập RUN_ROBERTA = False.")
elif _is_completed("roberta"):
    print("⏭️  Run 'roberta' đã completed, bỏ qua.")
else:
    run_cmd("python -m src.train --config configs/train_roberta.yaml")
print("✅ Xong bước RoBERTa")

CELL 5B — Phase 5.2: Train RoBERTa (3 epochs)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2681.25it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  epoch 1 step 250/250+ loss=0.2717 (4.0 it/s)
  epoch 1 step 500/500+ loss=0.1863 (4.0 it/s)
  epoch 1 step 750/750+ loss=0.1082 (4.1 it/s)
  epoch 1 step 1000/1000+ loss=0.0964 (4.1 it/s)
  epoch 1 step 1250/1250+ loss=0.1572 (4.1 it/s)
  epoch 1 step 1500/1500+ loss=0.2654 (4.1 it/s)
  epoch 1 step 1750/1750+ loss=0.4592 (4.1 it/s)
  epoch 1 step 2000/2000+ loss=0.1754 (4.1 it/s)
  e

## 6. Phase 5b — Dự đoán Test 1 lần, Xuất `model_summary.csv` & Kiểm thử tích hợp (Mốc 2)

1. Suy luận Test từng mô hình Transformer (`bert`, `distilbert`, và `roberta` nếu có) đúng 1 lần qua Python API của `src/predict.py`.
2. Ghép dự đoán với `baseline_test.csv`, bổ sung `n_tokens` và `split_id`.
3. Xây dựng `model_summary.csv` đủ 5 dòng theo thứ tự cố định: `mean_predictor`, `baseline`, `bert`, `distilbert`, `roberta`.
4. Chạy `scripts/validate_outputs.py` để xác nhận **`ALL CHECKS PASSED`** và liệt kê toàn bộ file Output.


In [10]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd

from src.baseline import load_baseline
from src.data_setup import create_dataloaders
from src.metrics import (
    compute_metrics,
    mean_predictor_metrics,
    per_star_metrics,
    rmse,
    write_model_summary,
)
from src.model_builder import build_tokenizer
from src.predict import (
    collect_summary_inputs,
    export_test_predictions,
    load_trained_model,
    merge_test_predictions,
)

print("=" * 60)
print("CELL 6 — Phase 5b: Predict Test, model_summary.csv & validate_outputs")
print("=" * 60)

decision_path = Path("/kaggle/working/results/phase4_decision.json")
if "BERT_WINNER_RUN" not in globals():
    if decision_path.exists():
        _dec = json.loads(decision_path.read_text(encoding="utf-8"))
        CHOSEN_LOSS = _dec["chosen_loss"]
        BERT_WINNER_RUN = _dec["bert_winner_run"]
        print(f"📦 Khôi phục quyết định Phase 4: BERT_WINNER_RUN='{BERT_WINNER_RUN}'")
    else:
        raise NameError("Chưa có quyết định Phase 4. Hãy chạy Cell 4 trước!")

DATA_DIR = "/kaggle/working/data"
CKPT_DIR = Path("/kaggle/working/checkpoints")
RES_DIR = Path("/kaggle/working/results")
PREDS_DIR = RES_DIR / "preds"

split_stats = json.loads((Path(DATA_DIR) / "split_stats.json").read_text(encoding="utf-8"))
SPLIT_ID = split_stats["split_id"]

# 1. Xác định các mô hình Transformer đã hoàn tất
model_runs = {
    "bert": BERT_WINNER_RUN,
    "distilbert": "distilbert",
}
if (CKPT_DIR / "roberta" / "best_model.pt").exists():
    model_runs["roberta"] = "roberta"
print(f"📋 Các mô hình Transformer sẽ dự đoán: {list(model_runs.keys())}")

# 2. Suy luận trên tập Test đúng 1 lần cho từng mô hình (bỏ qua nếu đã có file *_test.csv)
for model_name, run_name in model_runs.items():
    pred_csv = PREDS_DIR / f"{model_name}_test.csv"
    if pred_csv.exists():
        print(f"⏭️  Đã có sẵn {pred_csv.name}, bỏ qua bước suy luận GPU.")
        continue
    print(f"\n🔮 Đang xuất dự đoán Test cho {model_name} (từ run '{run_name}')...")
    tok = build_tokenizer(model_name)
    _, _, test_loader = create_dataloaders(DATA_DIR, tok, batch_size=32, num_workers=0)
    model, info = load_trained_model(str(CKPT_DIR / run_name / "best_model.pt"), device="cuda")
    info["split_id"] = SPLIT_ID
    export_test_predictions(
        model,
        test_loader,
        model_name=model_name,
        out_dir=str(PREDS_DIR),
        device="cuda",
        run_info=info,
        overwrite=True,
    )
    del model
    print(f"   ✅ {model_name} hoàn tất")

# Vá trực tiếp cột split_id cho toàn bộ các file *_test.csv trước khi ghép (tránh lỗi nan != nan)
for fname in ["baseline_test.csv", "bert_test.csv", "distilbert_test.csv", "roberta_test.csv"]:
    fpath = PREDS_DIR / fname
    if fpath.exists():
        df_tmp = pd.read_csv(fpath)
        df_tmp["split_id"] = SPLIT_ID
        df_tmp.to_csv(fpath, index=False, encoding="utf-8")
        print(f"   🔧 Đã gắn split_id='{SPLIT_ID}' cho {fname}")

# 3. Ghép dự đoán Test (kèm baseline_test.csv và cột n_tokens, split_id)
print("\n🔗 Đang ghép dự đoán Test...")
merged_df = merge_test_predictions(
    model_names=list(model_runs.keys()),
    preds_dir=str(PREDS_DIR),
    test_parquet=f"{DATA_DIR}/test.parquet",
    out_dir=str(RES_DIR),
    baseline_preds=str(PREDS_DIR / "baseline_test.csv"),
    overwrite=True,
)
print(f"✅ Đã tạo test_predictions.csv và test_predictions_with_text.csv! Số dòng: {len(merged_df)}")

# 4. Xây dựng bảng model_summary.csv đủ 5 dòng theo đúng thứ tự & schema
print("\n📊 Đang xây dựng model_summary.csv...")
train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test.parquet")
y_true_test = merged_df["true_stars"].to_numpy(dtype=float)
train_mean = float(train_df["stars"].mean())

summary_inputs = collect_summary_inputs(model_runs, str(RES_DIR))

# Dòng 1: mean_predictor
mp_m = mean_predictor_metrics(y_true_test, train_mean)
mp_ps = per_star_metrics(y_true_test, np.full_like(y_true_test, train_mean))
rows = [{
    "model": "mean_predictor",
    "mae": mp_m["mae"],
    "rmse": mp_m["rmse"],
    "rmse_raw": mp_m["rmse"],
    "r2": np.nan,
    "pearson": np.nan,
    "spearman": np.nan,
    "macro_mae": mp_ps["macro_mae"],
    "train_seconds": np.nan,
    "seconds_per_epoch": np.nan,
    "params_total": np.nan,
    "rmse_reduction_vs_baseline_pct": np.nan,
    "inference_seconds": np.nan,
    "split_id": SPLIT_ID,
}]

# Dòng 2: baseline
base_art = load_baseline(str(PREDS_DIR / "baseline_val.pkl"))
t0_base = time.perf_counter()
base_raw_test = base_art["pipeline"].predict(test_df["text"].astype(str).tolist())
base_infer_sec = time.perf_counter() - t0_base
base_clip_test = merged_df["pred_baseline"].to_numpy(dtype=float)

base_m = compute_metrics(y_true_test, base_clip_test, clip=True)
base_ps = per_star_metrics(y_true_test, base_clip_test, clip=True)
base_rmse = base_m["rmse"]
base_n_params = int(base_art["pipeline"].named_steps["ridge"].coef_.size) + 1

rows.append({
    "model": "baseline",
    "mae": base_m["mae"],
    "rmse": base_rmse,
    "rmse_raw": float(rmse(y_true_test, base_raw_test, clip=False)),
    "r2": base_m["r2"],
    "pearson": base_m["pearson"],
    "spearman": base_m["spearman"],
    "macro_mae": base_ps["macro_mae"],
    "train_seconds": float(base_art["fit_seconds"]),
    "seconds_per_epoch": np.nan,
    "params_total": base_n_params,
    "rmse_reduction_vs_baseline_pct": np.nan,
    "inference_seconds": float(base_infer_sec),
    "split_id": SPLIT_ID,
})

# Dòng 3–5: bert, distilbert, roberta
for model_name in ["bert", "distilbert", "roberta"]:
    if model_name in model_runs:
        y_pred_m = merged_df[f"pred_{model_name}"].to_numpy(dtype=float)
        m = compute_metrics(y_true_test, y_pred_m, clip=True)
        ps = per_star_metrics(y_true_test, y_pred_m, clip=True)
        red_pct = 100.0 * (base_rmse - m["rmse"]) / base_rmse
        rows.append({
            "model": model_name,
            "mae": m["mae"],
            "rmse": m["rmse"],
            "rmse_raw": float(summary_inputs["rmse_raw_by_model"][model_name]),
            "r2": m["r2"],
            "pearson": m["pearson"],
            "spearman": m["spearman"],
            "macro_mae": ps["macro_mae"],
            "train_seconds": float(summary_inputs["train_seconds"][model_name]),
            "seconds_per_epoch": float(summary_inputs["seconds_per_epoch"][model_name]),
            "params_total": int(summary_inputs["param_counts"][model_name]),
            "rmse_reduction_vs_baseline_pct": float(red_pct),
            "inference_seconds": float(summary_inputs["inference_seconds"][model_name]),
            "split_id": SPLIT_ID,
        })
    else:
        rows.append({
            "model": model_name,
            "mae": np.nan, "rmse": np.nan, "rmse_raw": np.nan,
            "r2": np.nan, "pearson": np.nan, "spearman": np.nan, "macro_mae": np.nan,
            "train_seconds": np.nan, "seconds_per_epoch": np.nan, "params_total": np.nan,
            "rmse_reduction_vs_baseline_pct": np.nan, "inference_seconds": np.nan,
            "split_id": SPLIT_ID,
        })

summary_df = pd.DataFrame(rows)
write_model_summary(summary_df, str(RES_DIR / "model_summary.csv"), overwrite=True)
print("\n✅ model_summary.csv đã ghi!")
print(summary_df.to_string(index=False))

# 5. Chạy kiểm thử tích hợp tự động
print("\n🧪 Đang chạy validate_outputs.py...")
ret = run_cmd(
    "python -m scripts.validate_outputs --results-dir /kaggle/working/results --split-stats /kaggle/working/data/split_stats.json",
    check=False,
)
if ret != 0:
    raise RuntimeError("validate_outputs.py phát hiện lỗi! Kiểm tra chi tiết bảng VALIDATION SUMMARY ở trên.")

# 6. Kiểm kê danh sách file output sẵn sàng lưu / tải về
for d in ["/kaggle/working/results", "/kaggle/working/results/preds", "/kaggle/working/checkpoints"]:
    p = Path(d)
    print(f"\n=== {d} ===")
    if p.exists():
        for item in sorted(p.iterdir()):
            if item.is_file():
                print(f"  {item.name}  ({item.stat().st_size / 1024 / 1024:.2f} MB)")
            else:
                print(f"  {item.name}/")

print("\n✅ Phase 5b hoàn tất — Sẵn sàng Save Version & chuyển sang Phase 6!")

CELL 6 — Phase 5b: Predict Test, model_summary.csv & validate_outputs
📋 Các mô hình Transformer sẽ dự đoán: ['bert', 'distilbert', 'roberta']
⏭️  Đã có sẵn bert_test.csv, bỏ qua bước suy luận GPU.
⏭️  Đã có sẵn distilbert_test.csv, bỏ qua bước suy luận GPU.
⏭️  Đã có sẵn roberta_test.csv, bỏ qua bước suy luận GPU.
   🔧 Đã gắn split_id='f0e6776f8a1a' cho baseline_test.csv
   🔧 Đã gắn split_id='f0e6776f8a1a' cho bert_test.csv
   🔧 Đã gắn split_id='f0e6776f8a1a' cho distilbert_test.csv
   🔧 Đã gắn split_id='f0e6776f8a1a' cho roberta_test.csv

🔗 Đang ghép dự đoán Test...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


✅ Đã tạo test_predictions.csv và test_predictions_with_text.csv! Số dòng: 8999

📊 Đang xây dựng model_summary.csv...

✅ model_summary.csv đã ghi!
         model      mae     rmse  rmse_raw       r2  pearson  spearman  macro_mae  train_seconds  seconds_per_epoch  params_total  rmse_reduction_vs_baseline_pct  inference_seconds     split_id
mean_predictor 1.261615 1.478705  1.478705      NaN      NaN       NaN   1.349705            NaN                NaN           NaN                             NaN                NaN f0e6776f8a1a
      baseline 0.575839 0.776944  0.800360 0.723932 0.853481  0.781468   0.631966      15.252458                NaN       20001.0                             NaN           1.142661 f0e6776f8a1a
          bert 0.341089 0.560791  0.562757 0.856173 0.926670  0.861342   0.440476    1787.610766         595.870255   108892417.0                       27.820883          37.188091 f0e6776f8a1a
    distilbert 0.363299 0.569338  0.570484 0.851756 0.923846  0.855454   0.453

In [11]:
!zip -r /kaggle/working/backup_all.zip /kaggle/working/results /kaggle/working/checkpoints

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/test_predictions_with_text.csv (deflated 59%)
  adding: kaggle/working/results/phase4_decision.json (deflated 46%)
  adding: kaggle/working/results/model_summary.csv (deflated 47%)
  adding: kaggle/working/results/runs/ (stored 0%)
  adding: kaggle/working/results/runs/roberta/ (stored 0%)
  adding: kaggle/working/results/runs/roberta/config.yaml (deflated 39%)
  adding: kaggle/working/results/runs/roberta/history.csv (deflated 42%)
  adding: kaggle/working/results/runs/roberta/run_meta.json (deflated 42%)
  adding: kaggle/working/results/runs/short_bert/ (stored 0%)
  adding: kaggle/working/results/runs/short_bert/config.yaml (deflated 39%)
  adding: kaggle/working/results/runs/short_bert/history.csv (deflated 30%)
  adding: kaggle/working/results/runs/short_bert/run_meta.json (deflated 44%)
  adding: kaggle/working/results/runs/distilbert/ (stored 0%)
  adding: kaggle/working/results/runs/distilbert/config

## 7. Checklist lưu trữ sau khi chạy xong Cell 6

| Bước | Hành động |
| :--- | :--- |
| 1 | Khi `validate_outputs.py` in `ALL CHECKS PASSED`, bấm **Save Version → Save & Run All (Commit)** (hoặc tải trực tiếp từ tab Output của phiên hiện tại nếu đã chạy xong trong Interactive Session) |
| 2 | Sao lưu `/kaggle/working/results/` và `/kaggle/working/checkpoints/` về Google Drive `My Drive/topic/` |
| 3 | Copy 2 file nhẹ `test_predictions.csv` và `model_summary.csv` vào thư mục `reports/` của repo local rồi commit/push lên GitHub |
| 4 | Giữ `test_predictions_with_text.csv` ở local/Drive (theo `.gitignore`, không commit) |
| 5 | Sang Phase 6: chạy `streamlit run app/streamlit_dashboard.py` và `python -m scripts.error_analysis prepare --reports-dir reports` |
